# Task 097: poseidon_atm — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Exoplanet atmosphere retrieval using POSEIDON radiative transfer

**Paper**: POSEIDON: A Multidimensional Atmospheric Retrieval Code for Exoplanet Spectra
**Repository**: https://github.com/MartianColonist/POSEIDON

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 45.50 dB
- **SSIM**: N/A (1D spectrum)
- **Evaluation**: 1D transmission spectrum retrieval (PSNR + CC + parameter RE)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
POSEIDON-inspired — Exoplanet Atmospheric Retrieval Inverse Problem
====================================================================
Task: Recover atmospheric composition and temperature from a
      transmission spectrum of a transiting exoplanet.

Inverse Problem:
    Given an observed wavelength-dependent transit depth spectrum D(λ),
    retrieve the atmospheric mixing ratios of absorbing species
    (H₂O, CH₄, CO₂) and the isothermal temperature T, plus the
    reference planetary radius R_p.

Forward Model (Simplified Transmission Spectroscopy):
    1. Set up a 1D isothermal atmosphere with N_layers pressure layers
       from P_top to P_bottom (log-spaced).
    2. At each layer, compute number density via ideal gas law:
       n(P) = P / (k_B T)
    3. For each wavelength λ, compute absorption cross-section σ(λ)
       as a sum of Gaussian absorption bands for each species.
    4. Compute slant optical depth τ(λ, z) through the limb geometry.
    5. Integrate to get the effective planetary radius R_eff(λ).
    6. Transit depth D(λ) = (R_eff(λ) / R_star)²

Inverse Solver:
    scipy.optimize.differential_evolution (global) with
    L-BFGS-B local refinement.

Repo inspiration: https://github.com/MartianColonist/POSEIDON
Reference: MacDonald & Madhusudhan (2017), MNRAS.

Usage:
    /data/yjh/poseidon_atm_env/bin/python poseidon_atm_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_cross_section`, `compute_rayleigh`, `params_from_vector`, `vector_from_params`, `chi_squared`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Operator (Transmission Spectroscopy)
# ═══════════════════════════════════════════════════════════
def compute_cross_section(wavelengths, species_name):
    """
    Compute simplified absorption cross-section for a species.

    Uses sum of Gaussian absorption bands. Real opacity databases
    (e.g., ExoMol, HITRAN) have millions of lines; this simplified
    model captures the essential wavelength-dependent structure.

    Parameters
    ----------
    wavelengths : np.ndarray  Wavelength array [m].
    species_name : str  Species name ('H2O', 'CH4', 'CO2').

    Returns
    -------
    sigma : np.ndarray  Cross-section array [m²] of shape (N_wave,).
    """
    bands = SPECIES_BANDS[species_name]
    sigma = np.zeros_like(wavelengths)
    for center, width, peak in bands:
        sigma += peak * np.exp(-0.5 * ((wavelengths - center) / width) ** 2)
    return sigma

def compute_rayleigh(wavelengths):
    """
    Rayleigh scattering cross-section for H₂ (λ⁻⁴ dependence).
    """
    return SIGMA_RAY_REF * (WAV_RAY_REF / wavelengths) ** 4

# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver (Atmospheric Retrieval)
# ═══════════════════════════════════════════════════════════
def params_from_vector(x):
    """
    Convert parameter vector to dict.

    x = [T, log_X_H2O, log_X_CH4, log_X_CO2, R_p_factor]
    where R_p = R_p_factor × R_JUP
    """
    return {
        "T":         x[0],
        "log_X_H2O": x[1],
        "log_X_CH4": x[2],
        "log_X_CO2": x[3],
        "R_p":       x[4] * R_JUP,
    }

def vector_from_params(params):
    """
    Convert parameter dict to vector.
    """
    return np.array([
        params["T"],
        params["log_X_H2O"],
        params["log_X_CH4"],
        params["log_X_CO2"],
        params["R_p"] / R_JUP,
    ])

def chi_squared(x, wavelengths, spectrum_obs, sigma_obs):
    """
    Chi-squared cost function for atmospheric retrieval.
    """
    params = params_from_vector(x)
    model = forward_operator(params, wavelengths)
    return np.sum(((spectrum_obs - model) / sigma_obs) ** 2)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `forward_operator`, `generate_data`, `load_or_generate_data`

In [ ]:
def forward_operator(params, wavelengths):
    """
    Compute the transmission spectrum (transit depth vs wavelength).

    Implements a simplified version of the atmospheric transmission
    calculation as in POSEIDON/Exo-Transmit/petitRADTRANS:

    1. Build pressure-altitude grid assuming hydrostatic equilibrium.
    2. At each layer, compute number densities of absorbers.
    3. For each wavelength, compute slant optical depth through each
       annular ring of atmosphere.
    4. Integrate to get the effective transit radius R_eff(λ).
    5. Transit depth D(λ) = (R_eff(λ)/R_star)².

    Parameters
    ----------
    params : dict  Atmospheric parameters.
    wavelengths : np.ndarray  Wavelength array [m].

    Returns
    -------
    transit_depth : np.ndarray  Transit depth D(λ).
    """
    T = params["T"]
    X_H2O = 10.0 ** params["log_X_H2O"]
    X_CH4 = 10.0 ** params["log_X_CH4"]
    X_CO2 = 10.0 ** params["log_X_CO2"]
    R_p = params["R_p"]

    # Pressure grid (log-spaced, top to bottom)
    pressures = np.logspace(np.log10(P_TOP), np.log10(P_BOTTOM), N_LAYERS)

    # Scale height H = k_B T / (mu g)
    g = G * M_PLANET / R_p ** 2  # surface gravity
    H = K_B * T / (MU_ATM * g)

    # Altitude grid from hydrostatic equilibrium: z = -H ln(P/P_ref)
    P_ref = pressures[-1]  # reference at bottom
    altitudes = -H * np.log(pressures / P_ref)  # z=0 at bottom

    # Layer boundaries (midpoints between levels)
    alt_boundaries = np.zeros(N_LAYERS + 1)
    alt_boundaries[0] = altitudes[0] + 0.5 * (altitudes[0] - altitudes[1])
    alt_boundaries[-1] = altitudes[-1] - 0.5 * (altitudes[-2] - altitudes[-1])
    alt_boundaries[1:-1] = 0.5 * (altitudes[:-1] + altitudes[1:])
    dz = np.abs(np.diff(alt_boundaries))  # layer thicknesses

    # Number densities [m⁻³]
    n_total = pressures / (K_B * T)
    n_H2O = X_H2O * n_total
    n_CH4 = X_CH4 * n_total
    n_CO2 = X_CO2 * n_total

    # Cross-sections
    sigma_H2O = compute_cross_section(wavelengths, "H2O")
    sigma_CH4 = compute_cross_section(wavelengths, "CH4")
    sigma_CO2 = compute_cross_section(wavelengths, "CO2")
    sigma_ray = compute_rayleigh(wavelengths)

    # Effective radius calculation
    # For each layer j, the annulus at radius r_j = R_p + z_j
    # contributes an area of ~2π r_j dr_j × (1 - exp(-τ_j))
    # to the total blocking area
    r = R_p + altitudes  # radius of each layer center

    # Transit depth: D(λ) = [R_p² + 2∫ r(1-exp(-τ)) dr] / R_star²
    # We compute τ along slant path through each annulus
    transit_depth = np.zeros(len(wavelengths))

    for j in range(N_LAYERS):
        # Total extinction at layer j for each wavelength
        kappa = (n_H2O[j] * sigma_H2O +
                 n_CH4[j] * sigma_CH4 +
                 n_CO2[j] * sigma_CO2 +
                 n_total[j] * sigma_ray)  # shape (N_wave,)

        # Slant path length through annulus at impact parameter b = r[j]
        # ds ≈ 2 * sqrt(2 * r[j] * H) for isothermal atmosphere
        ds = 2.0 * np.sqrt(2.0 * r[j] * H)

        # Optical depth along slant
        tau = kappa * ds  # shape (N_wave,)

        # Contribution to effective area
        transit_depth += 2.0 * r[j] * dz[j] * (1.0 - np.exp(-tau))

    # Add opaque disk of planet
    transit_depth = (R_p ** 2 + transit_depth) / R_STAR ** 2

    return transit_depth

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def generate_data(wavelengths):
    """
    Generate synthetic observed transmission spectrum.

    Returns
    -------
    spectrum_obs : np.ndarray  Noisy transit depth.
    spectrum_clean : np.ndarray  Clean transit depth.
    sigma_obs : np.ndarray  Noise standard deviation per point.
    """
    rng = np.random.RandomState(SEED)
    spectrum_clean = forward_operator(GT_PARAMS, wavelengths)
    sigma_obs = NOISE_LEVEL * np.ones(len(wavelengths))
    noise = rng.normal(0, sigma_obs)
    spectrum_obs = spectrum_clean + noise
    return spectrum_obs, spectrum_clean, sigma_obs

def load_or_generate_data():
    """
    Load cached data or generate fresh.
    """
    wavelengths = np.linspace(WAV_MIN, WAV_MAX, N_WAVE)
    data_file = os.path.join(RESULTS_DIR, "observed_spectrum.npz")

    if os.path.exists(data_file):
        print("[DATA] Loading cached data ...")
        data = np.load(data_file)
        return (wavelengths, data["spectrum_obs"],
                data["spectrum_clean"], data["sigma_obs"])

    print("[DATA] Generating synthetic transmission spectrum ...")
    spectrum_obs, spectrum_clean, sigma_obs = generate_data(wavelengths)

    np.savez(data_file,
             wavelengths=wavelengths,
             spectrum_obs=spectrum_obs,
             spectrum_clean=spectrum_clean,
             sigma_obs=sigma_obs)
    print(f"[DATA] Saved → {data_file}")
    return wavelengths, spectrum_obs, spectrum_clean, sigma_obs

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `reconstruct`

In [ ]:
def reconstruct(wavelengths, spectrum_obs, sigma_obs):
    """
    Perform atmospheric retrieval via global + local optimization.

    Phase 1: Differential evolution for global search.
    Phase 2: L-BFGS-B refinement from DE solution.

    Returns
    -------
    fit_params : dict  Retrieved atmospheric parameters.
    spectrum_fit : np.ndarray  Model spectrum from retrieved parameters.
    """
    print("[RECON] Phase 1: Differential evolution (global search) ...")
    result_de = differential_evolution(
        chi_squared,
        bounds=PARAM_BOUNDS,
        args=(wavelengths, spectrum_obs, sigma_obs),
        seed=SEED,
        maxiter=300,
        tol=1e-8,
        polish=False,
        popsize=20,
    )
    print(f"  DE result: χ² = {result_de.fun:.2f}, success = {result_de.success}")

    print("[RECON] Phase 2: L-BFGS-B local refinement ...")
    result_local = minimize(
        chi_squared,
        x0=result_de.x,
        args=(wavelengths, spectrum_obs, sigma_obs),
        method="L-BFGS-B",
        bounds=PARAM_BOUNDS,
        options={"maxiter": 500, "ftol": 1e-12},
    )
    print(f"  L-BFGS-B result: χ² = {result_local.fun:.2f}, success = {result_local.success}")

    fit_params = params_from_vector(result_local.x)
    spectrum_fit = forward_operator(fit_params, wavelengths)

    print("\n  Retrieved parameters:")
    gt_vec = vector_from_params(GT_PARAMS)
    fit_vec = result_local.x
    for i, name in enumerate(PARAM_NAMES):
        print(f"    {name:12s}: GT = {gt_vec[i]:10.4f}, Fit = {fit_vec[i]:10.4f}")

    return fit_params, spectrum_fit

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_params, fit_params, spectrum_clean, spectrum_fit):
    """
    Compute spectrum-fit and parameter-recovery metrics.

    Metrics:
      - PSNR: Peak signal-to-noise ratio of spectrum fit
      - CC: Pearson correlation coefficient
      - RMSE: Root mean square error of spectrum
      - RE: Relative error (norm)
      - Parameter-level relative errors
    """
    from skimage.metrics import structural_similarity as ssim_fn

    residual = spectrum_clean - spectrum_fit
    mse = np.mean(residual ** 2)
    rmse = float(np.sqrt(mse))

    # CC
    cc = float(np.corrcoef(spectrum_clean, spectrum_fit)[0, 1])

    # PSNR
    data_range = spectrum_clean.max() - spectrum_clean.min()
    psnr = float(10.0 * np.log10(data_range ** 2 / max(mse, 1e-30)))

    # SSIM (tile 1D to 2D for skimage)
    tile_rows = 7
    a2d = np.tile(spectrum_clean, (tile_rows, 1))
    b2d = np.tile(spectrum_fit, (tile_rows, 1))
    ssim = float(ssim_fn(a2d, b2d, data_range=data_range, win_size=7))

    # Relative error
    norm_gt = np.linalg.norm(spectrum_clean)
    re = float(np.linalg.norm(residual) / max(norm_gt, 1e-12))

    # Parameter recovery metrics
    param_keys = ["T", "log_X_H2O", "log_X_CH4", "log_X_CO2"]
    param_metrics = {}
    for k in param_keys:
        gt_v = gt_params[k]
        fit_v = fit_params[k]
        param_metrics[f"gt_{k}"] = float(gt_v)
        param_metrics[f"fit_{k}"] = float(fit_v)
        param_metrics[f"abs_err_{k}"] = float(abs(gt_v - fit_v))
        if abs(gt_v) > 1e-12:
            param_metrics[f"rel_err_{k}_pct"] = float(
                abs(gt_v - fit_v) / abs(gt_v) * 100
            )

    # Also compare R_p
    gt_rp = gt_params["R_p"] / R_JUP
    fit_rp = fit_params["R_p"] / R_JUP
    param_metrics["gt_R_p_Rjup"] = float(gt_rp)
    param_metrics["fit_R_p_Rjup"] = float(fit_rp)
    param_metrics["rel_err_R_p_pct"] = float(abs(gt_rp - fit_rp) / gt_rp * 100)

    metrics = {
        "PSNR": psnr,
        "SSIM": ssim,
        "CC": cc,
        "RMSE": rmse,
        "RE": re,
        **param_metrics,
    }
    return metrics

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(wavelengths, spectrum_obs, spectrum_clean, spectrum_fit,
                      gt_params, fit_params, metrics, save_path):
    """
    Generate a 4-panel visualization of the retrieval results.
    """
    wav_um = wavelengths * 1e6  # convert to μm
    depth_ppm_obs = spectrum_obs * 1e6
    depth_ppm_clean = spectrum_clean * 1e6
    depth_ppm_fit = spectrum_fit * 1e6

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # (a) Transmission spectrum
    ax = axes[0, 0]
    ax.scatter(wav_um, depth_ppm_obs, s=3, c='gray', alpha=0.4, label='Noisy obs')
    ax.plot(wav_um, depth_ppm_clean, 'b-', lw=1.5, label='Ground truth')
    ax.plot(wav_um, depth_ppm_fit, 'r--', lw=1.5, label='Retrieved')
    ax.set_xlabel('Wavelength [μm]')
    ax.set_ylabel('Transit Depth [ppm]')
    ax.set_title('(a) Transmission Spectrum')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # (b) Residuals
    ax = axes[0, 1]
    res_ppm = (depth_ppm_clean - depth_ppm_fit)
    ax.plot(wav_um, res_ppm, 'g-', lw=0.8)
    ax.axhline(0, color='k', ls='--', lw=0.5)
    ax.set_xlabel('Wavelength [μm]')
    ax.set_ylabel('Residual [ppm]')
    ax.set_title(f'(b) Residuals  RMSE = {metrics["RMSE"]*1e6:.2f} ppm')
    ax.grid(True, alpha=0.3)

    # (c) Absorption feature zoom (1.0–3.0 μm)
    ax = axes[1, 0]
    mask = (wav_um >= 1.0) & (wav_um <= 3.5)
    ax.plot(wav_um[mask], depth_ppm_clean[mask], 'b-', lw=2, label='GT')
    ax.plot(wav_um[mask], depth_ppm_fit[mask], 'r--', lw=2, label='Retrieved')

    # Annotate absorption bands
    band_labels = [
        (1.4, 'H₂O'), (1.65, 'CH₄'), (1.85, 'H₂O'),
        (2.3, 'CH₄'), (2.7, 'H₂O'),
    ]
    ymin, ymax = ax.get_ylim()
    for bwav, blabel in band_labels:
        if 1.0 <= bwav <= 3.5:
            ax.axvline(bwav, color='purple', alpha=0.3, ls=':')
            ax.text(bwav, ymax - 0.05 * (ymax - ymin), blabel,
                    ha='center', va='top', fontsize=7, color='purple')
    ax.set_xlabel('Wavelength [μm]')
    ax.set_ylabel('Transit Depth [ppm]')
    ax.set_title('(c) Absorption Features (1–3.5 μm)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # (d) Retrieved vs GT parameters (bar chart)
    ax = axes[1, 1]
    keys = ["T", "log_X_H2O", "log_X_CH4", "log_X_CO2", "R_p/R_Jup"]
    gt_vals = [
        gt_params["T"],
        gt_params["log_X_H2O"],
        gt_params["log_X_CH4"],
        gt_params["log_X_CO2"],
        gt_params["R_p"] / R_JUP,
    ]
    fit_vals = [
        fit_params["T"],
        fit_params["log_X_H2O"],
        fit_params["log_X_CH4"],
        fit_params["log_X_CO2"],
        fit_params["R_p"] / R_JUP,
    ]
    # Normalize for visualization (different scales)
    x = np.arange(len(keys))
    w = 0.35
    ax.bar(x - w / 2, gt_vals, w, label='GT', color='steelblue')
    ax.bar(x + w / 2, fit_vals, w, label='Retrieved', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(keys, fontsize=8, rotation=15)
    ax.set_title('(d) Parameter Recovery')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(
        f"POSEIDON-inspired — Exoplanet Atmospheric Retrieval\n"
        f"PSNR = {metrics['PSNR']:.1f} dB  |  "
        f"SSIM = {metrics['SSIM']:.4f}  |  "
        f"CC = {metrics['CC']:.6f}  |  "
        f"RE = {metrics['RE']:.2e}",
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 65)
print("  POSEIDON-inspired — Exoplanet Atmospheric Retrieval")
print("=" * 65)

wavelengths, spectrum_obs, spectrum_clean, sigma_obs = load_or_generate_data()

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print(f"\n[INFO] Wavelength range: {WAV_MIN*1e6:.1f} – {WAV_MAX*1e6:.1f} μm")
print(f"[INFO] Number of spectral points: {N_WAVE}")
print(f"[INFO] Noise level: {NOISE_LEVEL*1e6:.0f} ppm")

print("\n[RECON] Starting atmospheric retrieval ...")
fit_params, spectrum_fit = reconstruct(wavelengths, spectrum_obs, sigma_obs)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n[EVAL] Computing metrics ...")
metrics = compute_metrics(GT_PARAMS, fit_params, spectrum_clean, spectrum_fit)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"  {k:30s} = {v:.6g}")
    else:
        print(f"  {k:30s} = {v}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Save outputs
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
np.save(os.path.join(RESULTS_DIR, "gt_output.npy"), spectrum_clean)
np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), spectrum_fit)
np.save(os.path.join(RESULTS_DIR, "measurements.npy"), spectrum_obs)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print(f"\n[SAVE] gt_output.npy      → {RESULTS_DIR}")
print(f"[SAVE] recon_output.npy   → {RESULTS_DIR}")
print(f"[SAVE] measurements.npy   → {RESULTS_DIR}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
visualize_results(
    wavelengths, spectrum_obs, spectrum_clean, spectrum_fit,
    GT_PARAMS, fit_params, metrics,
    os.path.join(RESULTS_DIR, "reconstruction_result.png"),
)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("\n" + "=" * 65)
print("  DONE — Atmospheric retrieval benchmark complete")
print("=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **poseidon_atm**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=45.50 dB, SSIM=N/A (1D spectrum))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: POSEIDON: A Multidimensional Atmospheric Retrieval Code for Exoplanet Spectra
- Repository: https://github.com/MartianColonist/POSEIDON
